# Семинар 2. Data Lakehouse на PySpark

In [1]:
import os
import sys
import json
import shutil
from pathlib import Path
from datetime import datetime, timezone

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

java21_home = Path("/Library/Java/JavaVirtualMachines/jdk-21.jdk/Contents/Home")
if java21_home.exists():
    os.environ["JAVA_HOME"] = str(java21_home)
    os.environ["PATH"] = str(java21_home / "bin") + os.pathsep + os.environ["PATH"]

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
OUT = PROJECT_ROOT / "notebooks" / "lakehouse_demo_output"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

iceberg_warehouse = OUT / "iceberg_spark_warehouse"
iceberg_runtime = "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1"

spark_builder = (
    SparkSession.builder
    .appName("etl-course-lakehouse-demo")
    .master("local[*]")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.showConsoleProgress", "true")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", str(iceberg_warehouse))
)

spark = configure_spark_with_delta_pip(
    spark_builder,
    extra_packages=[iceberg_runtime],
).getOrCreate()

# spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Output directory:", OUT)

:: loading settings :: url = jar:file:/Users/avmysh/projs/2026/etl/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/avmysh/.ivy2.5.2/cache
The jars for the packages stored in: /Users/avmysh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-feff5ddc-5d89-4955-a8c7-e9996be62359;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.2.0 in central
	found io.delta#delta-storage;4.2.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.1 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	fo

Spark version: 4.1.1
Spark UI: http://192.168.1.11:4040
Output directory: /Users/avmysh/projs/2026/etl/notebooks/lakehouse_demo_output


## 1. Генерируем события интернет-магазина

In [2]:
def pick_from(values, index_col):
    return F.element_at(F.array(*[F.lit(v) for v in values]), index_col.cast("int"))

cities = ["Moscow", "Berlin", "Tbilisi", "Yerevan", "Belgrade"]
categories = ["books", "electronics", "grocery", "sport", "home"]
channels = ["web", "mobile", "marketplace"]
statuses = ["paid", "paid", "paid", "shipped", "cancelled"]
base_epoch = F.unix_timestamp(F.lit("2026-01-01 00:00:00"))

users = (
    spark.range(1, 1001)
    .withColumnRenamed("id", "user_id")
    .withColumn("city", pick_from(cities, (F.col("user_id") % len(cities)) + 1))
    .withColumn("segment", pick_from(["new", "regular", "vip"], (F.col("user_id") % 3) + 1))
)

users.show(n=5)

+-------+--------+-------+
|user_id|    city|segment|
+-------+--------+-------+
|      1|  Berlin|regular|
|      2| Tbilisi|    vip|
|      3| Yerevan|    new|
|      4|Belgrade|regular|
|      5|  Moscow|    vip|
+-------+--------+-------+
only showing top 5 rows


In [3]:
products = (
    spark.range(1, 301)
    .withColumnRenamed("id", "product_id")
    .withColumn("category", pick_from(categories, (F.col("product_id") % len(categories)) + 1))
    .withColumn("product_name", F.concat(F.lit("product_"), F.col("product_id")))
    .withColumn("base_price", (F.lit(50) + (F.col("product_id") % 120) * F.lit(3.7)).cast("double"))
)

products.show(n=5)

+----------+-----------+------------+----------+
|product_id|   category|product_name|base_price|
+----------+-----------+------------+----------+
|         1|electronics|   product_1|      53.7|
|         2|    grocery|   product_2|      57.4|
|         3|      sport|   product_3|      61.1|
|         4|       home|   product_4|      64.8|
|         5|      books|   product_5|      68.5|
+----------+-----------+------------+----------+
only showing top 5 rows


In [4]:
orders_v1 = (
    spark.range(1, 5001)
    .withColumnRenamed("id", "order_id")
    .withColumn("user_id", ((F.col("order_id") * 17) % 1000 + 1).cast("long"))
    .withColumn("order_epoch", base_epoch + (F.col("order_id") % 60) * 86400 + (F.col("order_id") % 24) * 3600)
    .withColumn("order_ts", F.to_timestamp(F.from_unixtime("order_epoch")))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("status", pick_from(statuses, (F.col("order_id") % len(statuses)) + 1))
    .withColumn("channel", pick_from(channels, (F.col("order_id") % len(channels)) + 1))
    .withColumn("updated_at", F.col("order_ts"))
    .drop("order_epoch")
)

late_updates = (
    orders_v1
    .where(F.col("order_id") <= 120)
    .withColumn("status", F.when(F.col("order_id") % 4 == 0, F.lit("refunded")).otherwise(F.lit("cancelled")))
    .withColumn("updated_at", F.col("order_ts") + F.expr("interval 3 days"))
)

bad_orders = (
    orders_v1
    .where(F.col("order_id").between(200, 205))
    .withColumn("user_id", F.lit(None).cast("long"))
    .withColumn("updated_at", F.col("order_ts") + F.expr("interval 1 day"))
)

orders_raw = orders_v1.unionByName(late_updates).unionByName(bad_orders)

orders_raw.show(n=5)

+--------+-------+-------------------+----------+---------+-----------+-------------------+
|order_id|user_id|           order_ts|order_date|   status|    channel|         updated_at|
+--------+-------+-------------------+----------+---------+-----------+-------------------+
|       1|     18|2026-01-02 01:00:00|2026-01-02|     paid|     mobile|2026-01-02 01:00:00|
|       2|     35|2026-01-03 02:00:00|2026-01-03|     paid|marketplace|2026-01-03 02:00:00|
|       3|     52|2026-01-04 03:00:00|2026-01-04|  shipped|        web|2026-01-04 03:00:00|
|       4|     69|2026-01-05 04:00:00|2026-01-05|cancelled|     mobile|2026-01-05 04:00:00|
|       5|     86|2026-01-06 05:00:00|2026-01-06|     paid|marketplace|2026-01-06 05:00:00|
+--------+-------+-------------------+----------+---------+-----------+-------------------+
only showing top 5 rows


In [5]:
order_items = (
    spark.range(1, 12001)
    .withColumnRenamed("id", "item_id")
    .withColumn("order_id", ((F.col("item_id") * 13) % 5000 + 1).cast("long"))
    .withColumn("product_id", ((F.col("item_id") * 7) % 300 + 1).cast("long"))
    .withColumn("quantity", ((F.col("item_id") % 4) + 1).cast("int"))
    .withColumn("discount_pct", ((F.col("item_id") % 5) * F.lit(0.03)).cast("double"))
)

bad_items = (
    order_items
    .where(F.col("item_id").between(20, 30))
    .withColumn("product_id", F.when(F.col("item_id") % 2 == 0, F.lit(None).cast("long")).otherwise(F.lit(99999).cast("long")))
    .withColumn("quantity", F.when(F.col("item_id") % 3 == 0, F.lit(-1)).otherwise(F.col("quantity")))
)
order_items_raw = order_items.unionByName(bad_items)

order_items_raw.show(n=5)

+-------+--------+----------+--------+------------+
|item_id|order_id|product_id|quantity|discount_pct|
+-------+--------+----------+--------+------------+
|      1|      14|         8|       2|        0.03|
|      2|      27|        15|       3|        0.06|
|      3|      40|        22|       4|        0.09|
|      4|      53|        29|       1|        0.12|
|      5|      66|        36|       2|         0.0|
+-------+--------+----------+--------+------------+
only showing top 5 rows


In [6]:
spark.sparkContext.setJobDescription("00 generate source data")
print("users:", users.count())
print("products:", products.count())
print("orders_raw:", orders_raw.count())
print("order_items_raw:", order_items_raw.count())
orders_raw.orderBy("order_id", "updated_at").show(8, truncate=False)

users: 1000
products: 300
orders_raw: 5126
order_items_raw: 12011
+--------+-------+-------------------+----------+---------+-----------+-------------------+
|order_id|user_id|order_ts           |order_date|status   |channel    |updated_at         |
+--------+-------+-------------------+----------+---------+-----------+-------------------+
|1       |18     |2026-01-02 01:00:00|2026-01-02|paid     |mobile     |2026-01-02 01:00:00|
|1       |18     |2026-01-02 01:00:00|2026-01-02|cancelled|mobile     |2026-01-05 01:00:00|
|2       |35     |2026-01-03 02:00:00|2026-01-03|paid     |marketplace|2026-01-03 02:00:00|
|2       |35     |2026-01-03 02:00:00|2026-01-03|cancelled|marketplace|2026-01-06 02:00:00|
|3       |52     |2026-01-04 03:00:00|2026-01-04|shipped  |web        |2026-01-04 03:00:00|
|3       |52     |2026-01-04 03:00:00|2026-01-04|cancelled|web        |2026-01-07 03:00:00|
|4       |69     |2026-01-05 04:00:00|2026-01-05|cancelled|mobile     |2026-01-05 04:00:00|
|4       |69  

## 2. Bronze: сырой слой

Bronze хранит данные максимально близко к источнику.  
Важно сохранить воспроизводимость: что пришло, когда пришло, из какого файла пришло.

In [8]:
bronze

PosixPath('/Users/avmysh/projs/2026/etl/notebooks/lakehouse_demo_output/bronze')

In [7]:
bronze = OUT / "bronze"

spark.sparkContext.setJobDescription("01 write raw bronze CSV")
(
    orders_raw
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(bronze / "orders_csv"))
)
(
    order_items_raw
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(bronze / "order_items_csv"))
)
(
    users
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(bronze / "users_csv"))
)
(
    products
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(bronze / "products_csv"))
)

print("Bronze folders:")
for path in sorted(bronze.iterdir()):
    print(" -", path.name)

Bronze folders:
 - order_items_csv
 - orders_csv
 - products_csv
 - users_csv


In [9]:
order_schema = T.StructType([
    T.StructField("order_id", T.LongType(), False),
    T.StructField("user_id", T.LongType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("order_date", T.DateType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("channel", T.StringType(), False),
    T.StructField("updated_at", T.TimestampType(), False),
])

item_schema = T.StructType([
    T.StructField("item_id", T.LongType(), False),
    T.StructField("order_id", T.LongType(), False),
    T.StructField("product_id", T.LongType(), True),
    T.StructField("quantity", T.IntegerType(), False),
    T.StructField("discount_pct", T.DoubleType(), False),
])

user_schema = T.StructType([
    T.StructField("user_id", T.LongType(), False),
    T.StructField("city", T.StringType(), False),
    T.StructField("segment", T.StringType(), False),
])

product_schema = T.StructType([
    T.StructField("product_id", T.LongType(), False),
    T.StructField("category", T.StringType(), False),
    T.StructField("product_name", T.StringType(), False),
    T.StructField("base_price", T.DoubleType(), False),
])

spark.sparkContext.setJobDescription("02 read bronze with explicit schema")
orders_bronze = (
    spark
    .read
    .schema(order_schema)
    .option("header", True)
    .csv(str(bronze / "orders_csv"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)
items_bronze = (
    spark
    .read
    .schema(item_schema)
    .option("header", True)
    .csv(str(bronze / "order_items_csv"))
)
users_bronze = (
    spark
    .read
    .schema(user_schema)
    .option("header", True)
    .csv(str(bronze / "users_csv"))
)
products_bronze = (
    spark
    .read
    .schema(product_schema)
    .option("header", True)
    .csv(str(bronze / "products_csv"))
)

orders_bronze.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- ingested_at: timestamp (nullable = false)
 |-- source_file: string (nullable = false)



In [10]:
orders_bronze.show(n=5)

+--------+-------+-------------------+----------+---------+-----------+-------------------+--------------------+--------------------+
|order_id|user_id|           order_ts|order_date|   status|    channel|         updated_at|         ingested_at|         source_file|
+--------+-------+-------------------+----------+---------+-----------+-------------------+--------------------+--------------------+
|    4688|    697|2026-01-09 08:00:00|2026-01-09|  shipped|marketplace|2026-01-09 08:00:00|2026-04-21 16:46:...|file:///Users/avm...|
|    4689|    714|2026-01-10 09:00:00|2026-01-10|cancelled|        web|2026-01-10 09:00:00|2026-04-21 16:46:...|file:///Users/avm...|
|    4690|    731|2026-01-11 10:00:00|2026-01-11|     paid|     mobile|2026-01-11 10:00:00|2026-04-21 16:46:...|file:///Users/avm...|
|    4691|    748|2026-01-12 11:00:00|2026-01-12|     paid|marketplace|2026-01-12 11:00:00|2026-04-21 16:46:...|file:///Users/avm...|
|    4692|    765|2026-01-13 12:00:00|2026-01-13|     paid|   

## 3. Parquet против CSV

CSV удобен для обмена и ручного просмотра, но для аналитики обычно проигрывает Parquet: нет колоночного чтения, слабая типизация, больше парсинга. Сравним размер и посмотрим план чтения.

In [11]:
def folder_size_mb(path: Path) -> float:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1024 / 1024

def count_data_files(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.is_file() and p.suffix in {".csv", ".parquet"})

parquet_orders_path = OUT / "parquet" / "orders"
spark.sparkContext.setJobDescription("03 write orders parquet")
(
    orders_bronze
    .write
    .mode("overwrite")
    .parquet(str(parquet_orders_path))
)

csv_size = folder_size_mb(bronze / "orders_csv")
parquet_size = folder_size_mb(parquet_orders_path)
print(f"CSV size:     {csv_size:.3f} MB")
print(f"Parquet size: {parquet_size:.3f} MB")
print("CSV data files:", count_data_files(bronze / "orders_csv"))
print("Parquet data files:", count_data_files(parquet_orders_path))

26/04/21 16:46:51 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/04/21 16:46:51 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers


CSV size:     0.464 MB
Parquet size: 0.089 MB
CSV data files: 18
Parquet data files: 9


26/04/21 16:46:52 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [12]:
for path in sorted((bronze / 'orders_csv').iterdir()):
    print(" -", path.name)

 - ._SUCCESS.crc
 - .part-00000-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00001-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00002-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00003-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00004-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00005-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00006-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00007-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00008-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00009-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00010-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00011-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00012-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00013-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part-00014-a5a3529d-5a04-459b-affc-8fae019c28f2-c000.csv.crc
 - .part

In [13]:
for path in sorted(parquet_orders_path.iterdir()):
    print(" -", path.name)

 - ._SUCCESS.crc
 - .part-00000-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00001-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00002-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00003-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00004-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00005-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00006-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00007-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - .part-00008-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet.crc
 - _SUCCESS
 - part-00000-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet
 - part-00001-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet
 - part-00002-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet
 - part-00003-1e9af8a0-bb40-4691-9158-32de444a49e5-c000.snappy.parquet
 - 

In [14]:
spark.sparkContext.setJobDescription("04 parquet projection and filter")
parquet_filtered = (
    spark
    .read
    .parquet(str(parquet_orders_path))
    .select("order_id", "order_date", "status")
    .where(
        (F.col("order_date") == F.to_date(F.lit("2026-04-19"))) & 
        (F.col("status") == "paid")
    )
)
parquet_filtered.explain("formatted")
parquet_filtered.show(10, truncate=False)

== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [order_id#255L, order_date#258, status#259]
Batched: true
Location: InMemoryFileIndex [file:/Users/avmysh/projs/2026/etl/notebooks/lakehouse_demo_output/parquet/orders]
PushedFilters: [IsNotNull(order_date), IsNotNull(status), EqualTo(order_date,2026-04-19), EqualTo(status,paid)]
ReadSchema: struct<order_id:bigint,order_date:date,status:string>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [order_id#255L, order_date#258, status#259]

(3) Filter [codegen id : 1]
Input [3]: [order_id#255L, order_date#258, status#259]
Condition : (((isnotnull(order_date#258) AND isnotnull(status#259)) AND (order_date#258 = 2026-04-19)) AND (status#259 = paid))


+--------+----------+------+
|order_id|order_date|status|
+--------+----------+------+
+--------+----------+------+



## 4. Silver: очищенные сущности

Silver — слой, где появляются инженерные гарантии: актуальная запись на ключ, корректные типы, исключение мусора

In [15]:
order_window = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())

silver_orders = (
    orders_bronze
    .where(F.col("user_id").isNotNull())
    .withColumn("rn", F.row_number().over(order_window))
    .where(F.col("rn") == 1)
    .drop("rn")
)

silver_items = (
    items_bronze
    .where(F.col("product_id").isNotNull())
    .where(F.col("quantity") > 0)
    .join(products_bronze.select("product_id"), "product_id", "inner")
)

silver_users = users_bronze.dropDuplicates(["user_id"])
silver_products = products_bronze.dropDuplicates(["product_id"])

spark.sparkContext.setJobDescription("05 clean silver entities")
print("raw orders:", orders_bronze.count())
print("silver orders:", silver_orders.count())
print("raw items:", items_bronze.count())
print("silver items:", silver_items.count())
silver_orders.groupBy("status").count().orderBy("status").show()

raw orders: 5126
silver orders: 5000
raw items: 12011
silver items: 12000
+---------+-----+
|   status|count|
+---------+-----+
|cancelled| 1066|
|     paid| 2928|
| refunded|   30|
|  shipped|  976|
+---------+-----+



## 5. Partitioning и partition pruning

Запишем `silver_orders` по `order_date`.  
Затем прочитаем только один день и посмотрим в `explain`, что Spark видит `PartitionFilters`

In [16]:
silver_root = OUT / "silver"
orders_partitioned_path = silver_root / "orders_partitioned"

spark.sparkContext.setJobDescription("06 write partitioned silver orders")
(
    silver_orders
    .repartition("order_date")
    .write.mode("overwrite")
    .partitionBy("order_date")
    .parquet(str(orders_partitioned_path))
)

print("Примеры partition-директорий:")
for p in sorted(orders_partitioned_path.glob("order_date=*"))[:8]:
    print(" -", p.name)
print("Всего partitions:", len(list(orders_partitioned_path.glob("order_date=*"))))

Примеры partition-директорий:
 - order_date=2026-01-01
 - order_date=2026-01-02
 - order_date=2026-01-03
 - order_date=2026-01-04
 - order_date=2026-01-05
 - order_date=2026-01-06
 - order_date=2026-01-07
 - order_date=2026-01-08
Всего partitions: 60


In [17]:
spark.sparkContext.setJobDescription("07 demonstrate partition pruning")
one_day = (
    spark.read.parquet(str(orders_partitioned_path))
    .where(F.col("order_date") == F.to_date(F.lit("2026-01-15")))
)
one_day.explain("formatted")
print("Rows for one day:", one_day.count())
one_day.orderBy("order_id").show(5)

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [9]: [order_id#361L, user_id#362L, order_ts#363, status#364, channel#365, updated_at#366, ingested_at#367, source_file#368, order_date#369]
Batched: true
Location: InMemoryFileIndex [file:/Users/avmysh/projs/2026/etl/notebooks/lakehouse_demo_output/silver/orders_partitioned]
PartitionFilters: [isnotnull(order_date#369), (order_date#369 = 2026-01-15)]
ReadSchema: struct<order_id:bigint,user_id:bigint,order_ts:timestamp,status:string,channel:string,updated_at:timestamp,ingested_at:timestamp,source_file:string>

(2) ColumnarToRow [codegen id : 1]
Input [9]: [order_id#361L, user_id#362L, order_ts#363, status#364, channel#365, updated_at#366, ingested_at#367, source_file#368, order_date#369]


Rows for one day: 84
+--------+-------+-------------------+---------+-----------+-------------------+--------------------+--------------------+----------+
|order_id|user_id|           order_ts|   status|    channel

## 6. Small files и compaction

Spark может очень быстро создать слишком много файлов: например, из-за большого числа partitions, частых маленьких batch-ей или неудачного `repartition`.  
Для Data Lakehouse это реальная проблема производительности.

In [18]:
small_files_path = OUT / "small_files_problem" / "orders"
compacted_path = OUT / "small_files_compacted" / "orders"

spark.sparkContext.setJobDescription("08 write many small files")
silver_orders.repartition(48).write.mode("overwrite").parquet(str(small_files_path))

spark.sparkContext.setJobDescription("09 compact files")
spark.read.parquet(str(small_files_path)).coalesce(4).write.mode("overwrite").parquet(str(compacted_path))

print("Small-files dataset files:", count_data_files(small_files_path))
print("Compacted dataset files:", count_data_files(compacted_path))
print(f"Small-files size: {folder_size_mb(small_files_path):.3f} MB")
print(f"Compacted size:   {folder_size_mb(compacted_path):.3f} MB")

Small-files dataset files: 48
Compacted dataset files: 4
Small-files size: 0.275 MB
Compacted size:   0.076 MB


## 7. Schema evolution

Покажем ситуацию, когда первый batch пришёл со старой схемой, а второй batch добавил новую колонку.  
В table formats эта тема управляется строгими правилами; в обычном Parquet нужно явно понимать, что делает Spark.

In [21]:
!ls -lha {schema_evolution_path}

total 0
drwxr-xr-x  4 avmysh  staff   128B 21 апр.  17:07 .
drwxr-xr-x  3 avmysh  staff    96B 21 апр.  17:07 ..
drwxr-xr-x  6 avmysh  staff   192B 21 апр.  17:07 batch=2026-01-early
drwxr-xr-x  6 avmysh  staff   192B 21 апр.  17:07 batch=2026-01-late


In [19]:
schema_evolution_path = OUT / "schema_evolution" / "orders"

batch_v1 = silver_orders.where(F.col("order_date") < F.to_date(F.lit("2026-01-20"))).select(
    "order_id", "user_id", "order_ts", "order_date", "status", "updated_at"
)

batch_v2 = silver_orders.where(F.col("order_date") >= F.to_date(F.lit("2026-01-20"))).select(
    "order_id", "user_id", "order_ts", "order_date", "status", "updated_at", "channel"
)

spark.sparkContext.setJobDescription("10 write parquet batches with different schemas")
batch_v1.write.mode("overwrite").parquet(str(schema_evolution_path / "batch=2026-01-early"))
batch_v2.write.mode("overwrite").parquet(str(schema_evolution_path / "batch=2026-01-late"))

print("Schema without mergeSchema:")
spark.read.parquet(str(schema_evolution_path)).printSchema()

print("Schema with mergeSchema=true:")
merged_schema_df = spark.read.option("mergeSchema", "true").parquet(str(schema_evolution_path))
merged_schema_df.printSchema()
merged_schema_df.select("order_id", "status", "channel", "batch").orderBy("order_id").show(8, truncate=False)

Schema without mergeSchema:
root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- batch: string (nullable = true)

Schema with mergeSchema=true:
root
 |-- order_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- channel: string (nullable = true)
 |-- batch: string (nullable = true)

+--------+---------+-------+-------------+
|order_id|status   |channel|batch        |
+--------+---------+-------+-------------+
|1       |cancelled|NULL   |2026-01-early|
|2       |cancelled|NULL   |2026-01-early|
|3       |cancelled|NULL   |2026-01-early|
|4       |refunded |NULL   |2026-01-early|
|5       |cancelled|NULL   |2026-0

## 8. Мини-transaction log: учебная модель table format

Настоящие Delta Lake и Apache Iceberg намного сложнее.  
Hudi решает похожий класс задач, но в этом семинаре остаётся кратким концептуальным упоминанием.  
Но центральная идея проста: данные пишутся в файлы, а актуальная версия таблицы публикуется через metadata/log.  
Сделаем маленькую модель: каждая версия — отдельная папка с Parquet, а JSON-log говорит, какая версия существует.

In [25]:
table_root = OUT / "managed_orders_table"
log_root = table_root / "_log"
data_root = table_root / "data"
log_root.mkdir(parents=True, exist_ok=True)
data_root.mkdir(parents=True, exist_ok=True)

def commit_table(df, version: int, operation: str, comment: str):
    version_path = data_root / f"v{version:03d}"
    spark.sparkContext.setJobDescription(f"commit table version {version:03d}: {operation}")
    df.write.mode("overwrite").parquet(str(version_path))
    record = {
        "version": version,
        "operation": operation,
        "comment": comment,
        "path": str(version_path),
        "committed_at": datetime.now(timezone.utc).isoformat(),
    }
    with open(log_root / f"{version:020d}.json", "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    return record

def read_log():
    records = []
    for path in sorted(log_root.glob("*.json")):
        with open(path, "r", encoding="utf-8") as f:
            records.append(json.load(f))
    return records

def read_table(version=None):
    records = read_log()
    if not records:
        raise ValueError("No committed versions")
    if version is None:
        record = max(records, key=lambda r: r["version"])
    else:
        candidates = [r for r in records if r["version"] == version]
        if not candidates:
            raise ValueError(f"Version {version} not found")
        record = candidates[0]
    return spark.read.parquet(record["path"])

base_table = silver_orders.select("order_id", "user_id", "order_ts", "order_date", "status", "channel", "updated_at")
commit_table(base_table, 0, "CREATE", "initial silver orders snapshot")
read_log()

[{'version': 0,
  'operation': 'CREATE',
  'comment': 'initial silver orders snapshot',
  'path': '/Users/avmysh/projs/2026/etl/notebooks/lakehouse_demo_output/managed_orders_table/data/v000',
  'committed_at': '2026-04-21T14:14:20.204985+00:00'},
 {'version': 1,
  'operation': 'CREATE',
  'comment': 'upd',
  'path': '/Users/avmysh/projs/2026/etl/notebooks/lakehouse_demo_output/managed_orders_table/data/v001',
  'committed_at': '2026-04-21T14:14:02.924441+00:00'}]

## 9. Upsert как новая версия таблицы

Сымитируем late updates: часть заказов стала `refunded`.  
Вместо опасной перезаписи исходной папки создадим новую версию и зафиксируем её в log.

In [26]:
updates = (
    base_table
    .where(F.col("order_id").between(1, 80))
    .withColumn("status", F.lit("refunded"))
    .withColumn("updated_at", F.col("updated_at") + F.expr("interval 5 days"))
)

merge_window = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
merged_v1 = (
    read_table(0)
    .unionByName(updates)
    .withColumn("rn", F.row_number().over(merge_window))
    .where(F.col("rn") == 1)
    .drop("rn")
)

commit_table(merged_v1, 1, "MERGE", "apply refunds for orders 1..80")

print("Transaction log:")
for record in read_log():
    print(record["version"], record["operation"], record["comment"])

print("Version 0 statuses for first orders:")
read_table(0).where(F.col("order_id") <= 10).groupBy("status").count().show()

print("Latest version statuses for first orders:")
read_table().where(F.col("order_id") <= 10).groupBy("status").count().show()

Transaction log:
0 CREATE initial silver orders snapshot
1 MERGE apply refunds for orders 1..80
Version 0 statuses for first orders:
+---------+-----+
|   status|count|
+---------+-----+
| refunded|    2|
|cancelled|    8|
+---------+-----+

Latest version statuses for first orders:
+--------+-----+
|  status|count|
+--------+-----+
|refunded|   10|
+--------+-----+



## 10. Time travel и rollback-мышление

Теперь можно открыть старую версию и сравнить её с последней.  
Именно поэтому версионность полезна для расследований: мы перестаём гадать, что было в данных вчера.

In [27]:
v0 = read_table(0).select("order_id", F.col("status").alias("status_v0"))
v1 = read_table(1).select("order_id", F.col("status").alias("status_v1"))

changed = (
    v0.join(v1, "order_id")
    .where(F.col("status_v0") != F.col("status_v1"))
    .orderBy("order_id")
)

print("Changed rows:", changed.count())
changed.show(12, truncate=False)

Changed rows: 60
+--------+---------+---------+
|order_id|status_v0|status_v1|
+--------+---------+---------+
|1       |cancelled|refunded |
|2       |cancelled|refunded |
|3       |cancelled|refunded |
|5       |cancelled|refunded |
|6       |cancelled|refunded |
|7       |cancelled|refunded |
|9       |cancelled|refunded |
|10      |cancelled|refunded |
|11      |cancelled|refunded |
|13      |cancelled|refunded |
|14      |cancelled|refunded |
|15      |cancelled|refunded |
+--------+---------+---------+
only showing top 12 rows


## 11. Gold: витрины для потребителей

Gold-слой должен быть понятен потребителю.  

In [28]:
orders_current = read_table()

fact_sales = (
    silver_items.alias("i")
    .join(F.broadcast(silver_products).alias("p"), "product_id", "inner")
    .join(orders_current.alias("o"), "order_id", "inner")
    .join(F.broadcast(silver_users).alias("u"), "user_id", "inner")
    .where(~F.col("o.status").isin("cancelled", "refunded"))
    .withColumn("gross_revenue", F.col("quantity") * F.col("base_price"))
    .withColumn("net_revenue", F.col("gross_revenue") * (F.lit(1.0) - F.col("discount_pct")))
    .select(
        "order_id", "item_id", "user_id", "product_id", "order_date", "status", "channel",
        "city", "segment", "category", "quantity", "gross_revenue", "net_revenue"
    )
)

spark.sparkContext.setJobDescription("11 build fact sales")
fact_sales.cache()
print("Fact sales rows:", fact_sales.count())
fact_sales.show(5)

Fact sales rows: 9365
+--------+-------+-------+----------+----------+-------+-----------+-------+-------+-----------+--------+------------------+------------------+
|order_id|item_id|user_id|product_id|order_date| status|    channel|   city|segment|   category|quantity|     gross_revenue|       net_revenue|
+--------+-------+-------+----------+----------+-------+-----------+-------+-------+-----------+--------+------------------+------------------+
|    1277|  11252|    710|       165|2026-01-18|   paid|marketplace| Moscow|    vip|      books|       1|             216.5|            203.51|
|    1290|  11253|    931|       172|2026-01-31|   paid|        web| Berlin|regular|    grocery|       2|             484.8|           441.168|
|    1303|  11254|    152|       179|2026-02-13|shipped|     mobile|Tbilisi|    vip|       home|       3| 804.9000000000001| 708.3120000000001|
|    1316|  11255|    373|       186|2026-02-26|   paid|marketplace|Yerevan|regular|electronics|       4|1176.8000

In [29]:
daily_sales_mart = (
    fact_sales
    .groupBy("order_date")
    .agg(
        F.countDistinct("order_id").alias("orders_cnt"),
        F.count("item_id").alias("items_cnt"),
        F.round(F.sum("net_revenue"), 2).alias("revenue"),
        F.round(F.avg("net_revenue"), 2).alias("avg_item_revenue"),
    )
    .orderBy("order_date")
)

category_sales_mart = (
    fact_sales
    .groupBy("order_date", "category")
    .agg(
        F.countDistinct("order_id").alias("orders_cnt"),
        F.round(F.sum("net_revenue"), 2).alias("revenue"),
        F.sum("quantity").alias("items_qty"),
    )
)

gold_root = OUT / "gold"
spark.sparkContext.setJobDescription("12 write gold marts")
daily_sales_mart.write.mode("overwrite").parquet(str(gold_root / "daily_sales_mart"))
category_sales_mart.write.mode("overwrite").partitionBy("order_date").parquet(str(gold_root / "category_sales_mart"))

daily_sales_mart.show(10, truncate=False)

+----------+----------+---------+---------+----------------+
|order_date|orders_cnt|items_cnt|revenue  |avg_item_revenue|
+----------+----------+---------+---------+----------------+
|2026-01-01|81        |196      |158931.14|810.87          |
|2026-01-02|82        |195      |42217.5  |216.5           |
|2026-01-03|82        |198      |84833.12 |428.45          |
|2026-01-04|82        |197      |148133.3 |751.95          |
|2026-01-06|82        |199      |49603.28 |249.26          |
|2026-01-07|82        |195      |105932.0 |543.24          |
|2026-01-08|82        |196      |148461.72|757.46          |
|2026-01-09|82        |198      |182435.97|921.39          |
|2026-01-11|82        |199      |93987.71 |472.3           |
|2026-01-12|82        |196      |148125.6 |755.74          |
+----------+----------+---------+---------+----------------+
only showing top 10 rows


## 12. Quality checks

Проверки качества лучше запускать рядом с расчётом.  
Если проверка падает, пайплайн не должен публиковать витрину как корректную.

In [30]:
spark.sparkContext.setJobDescription("13 quality checks")

assert silver_orders.count() == silver_orders.select("order_id").distinct().count(), "order_id is not unique in silver_orders"
assert silver_items.where(F.col("product_id").isNull()).count() == 0, "silver_items has null product_id"
assert fact_sales.where(F.col("net_revenue") < 0).count() == 0, "negative revenue detected"
assert daily_sales_mart.count() > 0, "daily_sales_mart is empty"
assert category_sales_mart.where(F.col("category").isNull()).count() == 0, "category_sales_mart has null category"

checks = [
    ("silver_orders_unique_order_id", "OK"),
    ("silver_items_no_null_product_id", "OK"),
    ("fact_sales_non_negative_revenue", "OK"),
    ("daily_sales_mart_not_empty", "OK"),
    ("category_sales_mart_no_null_category", "OK"),
]
spark.createDataFrame(checks, ["check_name", "status"]).show(truncate=False)

+------------------------------------+------+
|check_name                          |status|
+------------------------------------+------+
|silver_orders_unique_order_id       |OK    |
|silver_items_no_null_product_id     |OK    |
|fact_sales_non_negative_revenue     |OK    |
|daily_sales_mart_not_empty          |OK    |
|category_sales_mart_no_null_category|OK    |
+------------------------------------+------+



In [25]:
# spark.stop()

## 14. Реальные table formats: Delta Lake и Apache Iceberg

До этого момента мы показывали учебную модель: Parquet-файлы плюс маленький JSON transaction log.  
Теперь сопоставим эту модель с двумя форматами, которые запускаются в нашем окружении: Delta Lake и Apache Iceberg.

### Delta Lake: transaction log и Spark-first подход

Delta Lake ближе всего к нашей учебной модели: рядом с Parquet-файлами находится директория `_delta_log`, где хранятся commit-ы таблицы.

Сейчас мы создадим настоящую Delta-таблицу, затем сделаем `MERGE`: обновим заказ `order_id=2` и вставим новый заказ `order_id=4`.

Что смотреть после выполнения:

- текущие данные таблицы после merge;
- `history()` с версиями `WRITE` и `MERGE`;
- физическую директорию `_delta_log` с JSON commit-файлами.


In [31]:
real_formats_root = OUT / "real_table_formats"
delta_path = real_formats_root / "delta_orders"
if delta_path.exists():
    shutil.rmtree(delta_path)

spark.sparkContext.setJobDescription("14 real Delta Lake: write table")
delta_orders = spark.createDataFrame(
    [
        (1, "books", 100.0),
        (2, "electronics", 250.0),
        (3, "books", 75.0),
    ],
    ["order_id", "category", "amount"],
)

delta_orders.write.format("delta").mode("overwrite").save(str(delta_path))

spark.sparkContext.setJobDescription("15 real Delta Lake: merge upsert")
delta_updates = spark.createDataFrame(
    [
        (2, "electronics", 300.0),
        (4, "grocery", 20.0),
    ],
    ["order_id", "category", "amount"],
)

(
    DeltaTable.forPath(spark, str(delta_path))
    .alias("target")
    .merge(delta_updates.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Delta table after MERGE:")
spark.read.format("delta").load(str(delta_path)).orderBy("order_id").show(truncate=False)

print("Delta history:")
DeltaTable.forPath(spark, str(delta_path)).history().select("version", "operation").show(truncate=False)

print("Delta log files:")
for file in sorted((delta_path / "_delta_log").glob("*.json")):
    print(" -", file.name)


26/04/21 17:24:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/21 17:24:36 WARN MapPartitionsRDD: RDD 554 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


Delta table after MERGE:
+--------+-----------+------+
|order_id|category   |amount|
+--------+-----------+------+
|1       |books      |100.0 |
|2       |electronics|300.0 |
|3       |books      |75.0  |
|4       |grocery    |20.0  |
+--------+-----------+------+

Delta history:
+-------+---------+
|version|operation|
+-------+---------+
|1      |MERGE    |
|0      |WRITE    |
+-------+---------+

Delta log files:
 - 00000000000000000000.json
 - 00000000000000000001.json


In [35]:
import json
from pprint import pprint

In [36]:
raw_json = !cat {delta_path}/_delta_log/00000000000000000000.json
for sub_json in raw_json:
    pprint(json.loads(sub_json))

{'commitInfo': {'engineInfo': 'Apache-Spark/4.1.1 Delta-Lake/4.2.0',
                'isBlindAppend': False,
                'isolationLevel': 'Serializable',
                'operation': 'WRITE',
                'operationMetrics': {'numFiles': '4',
                                     'numOutputBytes': '3568',
                                     'numOutputRows': '3',
                                     'numRemovedBytes': '0',
                                     'numRemovedFiles': '0'},
                'operationParameters': {'mode': 'Overwrite',
                                        'partitionBy': '[]'},
                'timestamp': 1776781474666,
                'txnId': 'cc168b9c-b960-4790-91bb-c841b974736f'}}
{'metaData': {'configuration': {},
              'createdTime': 1776781473986,
              'format': {'options': {}, 'provider': 'parquet'},
              'id': '304dca9e-f500-44ac-886a-1ee1f9b447e0',
              'partitionColumns': [],
              'schemaString': 

In [37]:
raw_json = !cat {delta_path}/_delta_log/00000000000000000001.json
for sub_json in raw_json:
    pprint(json.loads(sub_json))

{'commitInfo': {'engineInfo': 'Apache-Spark/4.1.1 Delta-Lake/4.2.0',
                'isBlindAppend': False,
                'isolationLevel': 'Serializable',
                'operation': 'MERGE',
                'operationMetrics': {'executionTimeMs': '940',
                                     'materializeSourceTimeMs': '266',
                                     'numOutputRows': '2',
                                     'numSourceRows': '2',
                                     'numTargetBytesAdded': '1050',
                                     'numTargetBytesRemoved': '1058',
                                     'numTargetChangeFilesAdded': '0',
                                     'numTargetDeletionVectorsAdded': '0',
                                     'numTargetDeletionVectorsRemoved': '0',
                                     'numTargetDeletionVectorsUpdated': '0',
                                     'numTargetFilesAdded': '1',
                                     'numTargetF

### Apache Iceberg: snapshots, manifests и multi-engine lakehouse

Iceberg делает большой акцент на metadata-структуре таблицы и работе через catalog.  
Таблица обычно адресуется не как путь к папке, а как имя в catalog: `catalog.database.table`.

Сейчас мы создадим настоящую Iceberg-таблицу в локальном Spark catalog `local`, добавим данные и посмотрим snapshot metadata.

In [38]:
iceberg_table = "local.demo.orders_real"

spark.sparkContext.setJobDescription("16 real Iceberg: create table")
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.demo")
spark.sql(f"DROP TABLE IF EXISTS {iceberg_table}")

iceberg_orders = spark.createDataFrame(
    [
        (1, "books", 100.0),
        (2, "electronics", 250.0),
        (3, "books", 75.0),
    ],
    ["order_id", "category", "amount"],
)
iceberg_orders.writeTo(iceberg_table).using("iceberg").create()

spark.sparkContext.setJobDescription("17 real Iceberg: append data")
iceberg_updates = spark.createDataFrame(
    [(4, "grocery", 20.0)],
    ["order_id", "category", "amount"],
)
iceberg_updates.writeTo(iceberg_table).append()

print("Iceberg table after append:")
spark.table(iceberg_table).orderBy("order_id").show(truncate=False)

print("Iceberg snapshots:")
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {iceberg_table}.snapshots").show(truncate=False)


Iceberg table after append:
+--------+-----------+------+
|order_id|category   |amount|
+--------+-----------+------+
|1       |books      |100.0 |
|2       |electronics|250.0 |
|3       |books      |75.0  |
|4       |grocery    |20.0  |
+--------+-----------+------+

Iceberg snapshots:
+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-04-21 17:31:22.094|4137846691174148627|append   |
|2026-04-21 17:31:22.455|8868449714202841737|append   |
+-----------------------+-------------------+---------+

